# The clock that ends on 19 January 2038

Many systems store time as **the number of seconds since 1 January 1970** in a
**32-bit signed integer**. This notebook shows exactly what happens at the last
second such a counter can hold, and what comes after.

In [1]:
from datetime import datetime, timezone

MAX32 = 2**31 - 1          # the largest value of a signed 32-bit integer
print("largest value:", MAX32)
print("as a date:    ", datetime.fromtimestamp(MAX32, timezone.utc))

largest value: 2147483647
as a date:     2038-01-19 03:14:07+00:00


## The bit pattern

Two's complement uses the first bit as the sign. The largest positive value is a
`0` followed by 31 ones.

In [2]:
print(format(MAX32, "032b"))

01111111111111111111111111111111


## Adding one second

Python integers never overflow, so we have to *emulate* a 32-bit integer: keep
only the low 32 bits, then interpret the result as signed. `struct` does exactly
that: pack as unsigned 32-bit, unpack as signed.

In [3]:
import struct

def as_int32(n):
    """Interpret the low 32 bits of n as a signed 32-bit integer."""
    return struct.unpack("<i", struct.pack("<I", n & 0xFFFFFFFF))[0]

after = as_int32(MAX32 + 1)
print("bits after +1: ", format((MAX32 + 1) & 0xFFFFFFFF, "032b"))
print("value:         ", after)
print("as a date:     ", datetime.fromtimestamp(after, timezone.utc))

bits after +1:  10000000000000000000000000000000
value:          -2147483648
as a date:      1901-12-13 20:45:52+00:00


The carry ran into the sign bit: `1000…0` is $-2^{31}$, and the clock jumps
back to **13 December 1901**.

## The fix: 64 bits

With a 64-bit counter the same overflow is $2^{63}$ seconds away.

In [4]:
seconds_per_year = 365.2425 * 24 * 3600
print(f"{(2**63 - 1) / seconds_per_year / 1e9:.0f} billion years")

292 billion years


**What we used:** positional number systems, two's complement, and the fact that
a data type has a fixed size (transcript sections 4, 6 and 10 of session 2).